In [ ]:
import base64
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage
from dotenv import load_dotenv
import os

load_dotenv(override=True)
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = "https://api.deepseek.com"
model = init_chat_model(
    model="deepseek-v4-flash",
    model_provider="deepseek",
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL
)


def encode_image(img_path, img_type='jpeg'):
    """将一张本地图片转换成 Base64 编码的 Data URI 字符串,方便在文本中嵌入图片数据"""
    with open(img_path, "rb") as img_file:
        return f"data:image/{img_type};base64,{base64.b64encode(img_file.read()).decode("utf-8")}"


# 图像路径
img_path = "image_test.png"
# 获取图像base64编码字符串
base64_image = encode_image(img_path)
response = model.invoke(
    [
        HumanMessage(
            content=[
                {'type': 'text', 'text': '这张图里有什么？'},
                {
                    'type': 'image_url',
                    "image_url": base64_image,
                }
            ]
        )
    ]
)
print(response.content)


1.7.2 content_blocks
在 LangChain 1.x 中，
content_blocks 是消息对象（BaseMessage）的一项重大升级。它的核心目标
是提供一种跨模型供应商、标准化的多模态数据结构。
过去，处理图片、音频、甚至是模型生成的“思维链（Reasoning）”内容时，不同供应商（OpenAI,
Anthropic, Google 等）的 API 格式各异，导致开发者需要写大量的适配代码。
content_blocks 的出
现终结了这种混乱。
在 LangChain 1.2 版本中，消息对象的
content 属性依然存在（为了向前兼容），但新增了
content_blocks 属性，可以将
content 解析为标准、类型安全的表示。
数据结构：它是一个
list[TypedDict] 。
统一格式：每个 block 都有一个
type 字段，用于区分内容类型。
支持类型：包括
text （文本）、
image （图片）、
tool_call （工具调用）以及
audio （音频）、
video （视频）、
reasoning （推理/思维链）。
支持的字段类型详见
https://docs.langchain.com/oss/python/langchain/messages#openai
① 输入格式化
对于复杂的对话（带图片或工具结果），建议使用
或 AIMessage。
借助
content_blocks 列表形式构建
HumanMessage
content_blocks ，我们可以用一套标准代码，无缝地在不同厂商的模型之间切换。

In [ ]:
from langchain.messages import HumanMessage
import os
from dotenv import load_dotenv

load_dotenv(override=True)
model = init_chat_model(
    model="gpt-5.4-mini",
    model_provider="openai",
    api_key=os.getenv("CLOSEAI_API_KEY"),
    base_url=os.getenv("CLOSEAI_BASE_URL")
)


def encode_image(img_path):
    """将一张本地图片转换成 Base64 编码的 Data URI 字符串,方便在文本中嵌入图片数据"""
    with open(img_path, "rb") as img_file:
        return base64.b64encode(img_file.read()).decode("utf-8")


# 图像路径
img_path = "image_test.png"
# 获取图像base64编码字符串
base64_image = encode_image(img_path)
response = model.invoke(
    [
        # 此种格式可用
        # HumanMessage(
        #     content=[
        #         {'type': 'text', 'text': '这张图里有什么？'},
        #         {'type': 'image_url', "image_url": base64_image}
        #     ]
        # )
        # 推荐的统一写法
        HumanMessage(
            content_blocks=[
                {'type': 'text', 'text': '这张图里有什么？'},
                {
                    'type': 'image',
                    'base64': base64_image,
                    'mime_type': 'image/png',
                }
            ]
        )
    ])

print(response.content)


输出格式化
不同的模型其输出格式可能不同，仅为提取思考内容，切换模型都可能需要更改代码，非常不方便。
content_blocks提供了统一的输出格式，可以将不同格式的响应统一为标准格式。
注意：content_blocks是懒加载的，即调用时才会解析。

In [1]:
from langchain.chat_models import init_chat_model

from dotenv import load_dotenv

load_dotenv(override=True)
model = init_chat_model(
    model="deepseek:deepseek-v4-flash",
    extra_body={"thinking": {"type": "enabled"}},
)
response = model.invoke("你好，一句话回答")
print('=' * 20, '-> response <-', '=' * 20)
print(response)
print('=' * 20, '-> response.content <-', '=' * 20)
print(response.content)
print('=' * 20, '-> response.content_blocks <-', '=' * 20)
print(response.content_blocks)

==================== -> response <- ====================
content='你好，有什么可以帮你的？' additional_kwargs={'refusal': None, 'reasoning_content': '好的，用户让我用一句话回答。既然用户没有提出具体问题，只说了“你好，一句话回答”，这可能是打招呼并明确要求回复格式。我需要遵守指令，用一句简洁的话回应问候，同时表明准备就绪的状态。可以用“你好，有什么可以帮你的？”这样既符合一句话的要求，又自然开启对话。'} response_metadata={'token_usage': {'completion_tokens': 78, 'prompt_tokens': 8, 'total_tokens': 86, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 70, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 8}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402', 'id': 'f1754b19-378a-4ba0-b542-ee3a61951330', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019f4a1f-9a72-7230-901c-a0ac15dfc751-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 8, 'outpu